# 04 — Biological Sequence Files with Biopython

> **A practical guide to reading, inspecting, filtering, converting, and summarizing biological sequence files**

This notebook continues the sequence of lessons:

1. **Python for Bioinformatics** — Python foundations in biological contexts.
2. **Practical DNA & RNA Sequence Analysis with Python** — sequence-analysis logic using plain Python.
3. **Biopython Fundamentals** — `Seq`, `MutableSeq`, and `SeqRecord`.
4. **Biological Sequence Files with Biopython** — working with collections of records and common biological file formats.

The main focus is `Bio.SeqIO`, Biopython's interface for reading and writing sequence files.

## Learning objectives

By the end of this notebook, you should be able to:

- Explain the difference between a sequence and a sequence record.
- Read FASTA and GenBank files with `SeqIO`.
- Inspect sequence IDs, descriptions, lengths, and annotations.
- Iterate through records without loading more data than necessary.
- Filter records by length and simple sequence-quality criteria.
- Write selected records to FASTA and convert supported formats.
- Calculate a compact summary of a sequence collection.
- Build a reusable mini-pipeline that produces filtered FASTA and a CSV report.

> **Data note:** The notebook creates small synthetic records so the core lesson runs offline and does not depend on downloading files. Synthetic sequences are for teaching and do not represent real organisms, samples, or patients.

## 1. Setup and imports

Install Biopython if needed from a terminal:

```bash
python -m pip install biopython
```

If you are using Jupyter, you can run the command in a notebook cell by prefixing it with `!`, but installing from the environment's terminal is often easier to troubleshoot.

In [2]:
from pathlib import Path
from collections import Counter
import csv
import Bio
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

print("Biopython version:", Bio.__version__)

Biopython version: 1.88


## 2. The key idea: `Seq` versus `SeqRecord`

A `Seq` stores biological sequence data. A `SeqRecord` wraps a sequence with record-level information such as an identifier, a name, a description, and optionally annotations or features.

| Object or field | Purpose |
|---|---|
| `record.seq` | The sequence itself |
| `record.id` | The record identifier |
| `record.name` | A short name |
| `record.description` | A human-readable description |
| `record.annotations` | Additional metadata, when available |
| `record.features` | Annotated biological features, when available |

The exact metadata available depends on the input file format. A FASTA record generally contains an identifier and description, while a GenBank record can contain richer annotations and features.

In [3]:
record = SeqRecord(
    Seq("ATGCGTACGTTAGC"),
    id="synthetic_gene_01",
    name="gene_01",
    description="Synthetic example sequence for learning SeqRecord"
)

print("ID:", record.id)
print("Name:", record.name)
print("Description:", record.description)
print("Sequence:", record.seq)
print("Length:", len(record.seq))
print("Annotations:", record.annotations)

ID: synthetic_gene_01
Name: gene_01
Description: Synthetic example sequence for learning SeqRecord
Sequence: ATGCGTACGTTAGC
Length: 14
Annotations: {}


## 3. Create a small FASTA file

FASTA is a widely used text format for biological sequences. A record begins with a header line starting with `>`; the sequence follows on one or more lines.

We will create a local example file so that the notebook is reproducible and can run without internet access.

**You can also obtain real biological sequence data from NCBI (National Center for Biotechnology Information).** NCBI provides public nucleotide and protein sequences that can be downloaded in FASTA format. For example, you can search for a gene or organism in [NCBI Nucleotide](https://www.ncbi.nlm.nih.gov/nuccore/) and download its sequence.

For real datasets, you can also retrieve sequences programmatically using Biopython's `Bio.Entrez` module. This allows you to fetch records directly from NCBI instead of creating example files manually. We will explore this approach in a later notebook.

> **Note:** The local FASTA file used in this section contains example data and does not require an internet connection. NCBI downloads contain real biological records and require internet access.

In [16]:
data_dir = Path("data")
output_dir = Path("outputs")
data_dir.mkdir(exist_ok=True)
output_dir.mkdir(exist_ok=True)

fasta_path = data_dir / "synthetic_sequences.fasta"

records = [
    SeqRecord(
        Seq("ATGCGTACGTAG"),
        id="SEQ001",
        description="synthetic DNA example A"
    ),
    SeqRecord(
        Seq("ATGAAACCCGGGTTT"),
        id="SEQ002",
        description="synthetic DNA example B"
    ),
    SeqRecord(
        Seq("NNATGCCGTANNN"),
        id="SEQ003",
        description="synthetic sequence containing ambiguous bases"
    ),
    SeqRecord(
        Seq("ATGC"),
        id="SEQ004",
        description="short synthetic sequence"
    ),
    SeqRecord(
        Seq("ATGCGCGCGCGCTAA"),
        id="SEQ005",
        description="GC-rich synthetic DNA example"
    ),
]

SeqIO.write(records, fasta_path, "fasta")
print(f"Wrote {len(records)} records to {fasta_path}")

Wrote 5 records to data\synthetic_sequences.fasta


## 4. Read FASTA with `SeqIO.parse`

`SeqIO.parse()` returns an iterator of `SeqRecord` objects. It is a good default for reading files that may contain multiple records.

Do not assume the file has only one sequence. Always inspect the number of records and their identifiers when starting a new dataset.

In [17]:
parsed_records = list(SeqIO.parse(fasta_path, "fasta"))

print("Number of records:", len(parsed_records))

for record in parsed_records:
    print(record.id, len(record.seq), record.description)

Number of records: 5
SEQ001 12 SEQ001 synthetic DNA example A
SEQ002 15 SEQ002 synthetic DNA example B
SEQ003 13 SEQ003 synthetic sequence containing ambiguous bases
SEQ004 4 SEQ004 short synthetic sequence
SEQ005 15 SEQ005 GC-rich synthetic DNA example


### `parse()` versus `read()`

- Use `SeqIO.parse(handle, format)` when a file may contain **zero, one, or many** records.
- Use `SeqIO.read(handle, format)` when the file is expected to contain **exactly one** record. It raises an error if the record count is not exactly one.

For large files, prefer iterating directly over `SeqIO.parse()` instead of converting the entire iterator to a list.

In [18]:
# Demonstrate the single-record API using an in-memory record.
from io import StringIO

one_record_fasta = StringIO(">one_record example\nATGCGT\n")
single_record = SeqIO.read(one_record_fasta, "fasta")

print(single_record.id)
print(single_record.seq)

one_record
ATGCGT


## 5. Inspect record metadata and sequence content

FASTA headers are not a complete substitute for a structured annotation format. The `id` is commonly the first token in the header, while the description preserves the header text parsed by Biopython.

In [19]:
for record in SeqIO.parse(fasta_path, "fasta"):
    sequence = str(record.seq).upper()
    print({
        "id": record.id,
        "description": record.description,
        "length": len(sequence),
        "alphabet_preview": "".join(sorted(set(sequence)))
    })

{'id': 'SEQ001', 'description': 'SEQ001 synthetic DNA example A', 'length': 12, 'alphabet_preview': 'ACGT'}
{'id': 'SEQ002', 'description': 'SEQ002 synthetic DNA example B', 'length': 15, 'alphabet_preview': 'ACGT'}
{'id': 'SEQ003', 'description': 'SEQ003 synthetic sequence containing ambiguous bases', 'length': 13, 'alphabet_preview': 'ACGNT'}
{'id': 'SEQ004', 'description': 'SEQ004 short synthetic sequence', 'length': 4, 'alphabet_preview': 'ACGT'}
{'id': 'SEQ005', 'description': 'SEQ005 GC-rich synthetic DNA example', 'length': 15, 'alphabet_preview': 'ACGT'}


## 6. Filter records by length and ambiguous bases

Filtering thresholds should be chosen for the biological question and data source. The thresholds below are only teaching examples, not universal quality-control rules.

Here, a record passes if it is at least 10 nucleotides long and its sequence contains only canonical DNA bases (`A`, `C`, `G`, `T`). Sequences containing `N` are treated as ambiguous and excluded from this particular filtered output.

In [20]:
CANONICAL_DNA = set("ACGT")

def sequence_qc(sequence, min_length=10):
    """Return basic, transparent QC metrics for a DNA sequence."""
    seq = str(sequence).upper()
    invalid_symbols = sorted(set(seq) - CANONICAL_DNA)
    return {
        "length": len(seq),
        "valid_symbols": len(invalid_symbols) == 0,
        "invalid_symbols": invalid_symbols,
        "passes_length": len(seq) >= min_length,
        "passes": len(seq) >= min_length and len(invalid_symbols) == 0,
    }

for record in parsed_records:
    print(record.id, sequence_qc(record.seq, min_length=10))

SEQ001 {'length': 12, 'valid_symbols': True, 'invalid_symbols': [], 'passes_length': True, 'passes': True}
SEQ002 {'length': 15, 'valid_symbols': True, 'invalid_symbols': [], 'passes_length': True, 'passes': True}
SEQ003 {'length': 13, 'valid_symbols': False, 'invalid_symbols': ['N'], 'passes_length': True, 'passes': False}
SEQ004 {'length': 4, 'valid_symbols': True, 'invalid_symbols': [], 'passes_length': False, 'passes': False}
SEQ005 {'length': 15, 'valid_symbols': True, 'invalid_symbols': [], 'passes_length': True, 'passes': True}


## 7. Filter and write selected records

`SeqIO.write()` writes one or more `SeqRecord` objects to a file and returns the number of records written.

We will keep the original records unchanged and create a separate output file for the selected subset. This preserves the source data and makes the workflow easier to audit.

In [21]:
min_length = 10
filtered_records = []

for record in SeqIO.parse(fasta_path, "fasta"):
    qc = sequence_qc(record.seq, min_length=min_length)
    if qc["passes"]:
        filtered_records.append(record)

filtered_fasta_path = output_dir / "filtered_sequences.fasta"
written_count = SeqIO.write(filtered_records, filtered_fasta_path, "fasta")

print("Records passing filters:", len(filtered_records))
print("Records written:", written_count)
print("Output file:", filtered_fasta_path)

Records passing filters: 3
Records written: 3
Output file: outputs\filtered_sequences.fasta


## 8. Summarize a sequence collection

A useful first-pass report can include record count, total sequence length, minimum and maximum lengths, mean length, and GC content.

GC content is calculated here only for canonical bases. If a sequence contains ambiguous symbols, the denominator is the number of `A`, `C`, `G`, and `T` bases rather than the total length. This convention should always be stated when reporting results.

In [22]:
def collection_summary(records):
    """Summarize lengths and GC content across SeqRecord objects."""
    lengths = []
    total_a = total_c = total_g = total_t = 0

    for record in records:
        seq = str(record.seq).upper()
        lengths.append(len(seq))
        total_a += seq.count("A")
        total_c += seq.count("C")
        total_g += seq.count("G")
        total_t += seq.count("T")

    canonical_total = total_a + total_c + total_g + total_t
    gc_percent = (
        100 * (total_g + total_c) / canonical_total
        if canonical_total else None
    )

    return {
        "record_count": len(records),
        "total_length": sum(lengths),
        "min_length": min(lengths) if lengths else None,
        "max_length": max(lengths) if lengths else None,
        "mean_length": sum(lengths) / len(lengths) if lengths else None,
        "canonical_bases": canonical_total,
        "GC_percent_of_canonical_bases": gc_percent,
    }

summary = collection_summary(parsed_records)
for key, value in summary.items():
    print(f"{key}: {value}")

record_count: 5
total_length: 59
min_length: 4
max_length: 15
mean_length: 11.8
canonical_bases: 54
GC_percent_of_canonical_bases: 53.7037037037037


## 9. Export a per-record CSV report

CSV is convenient when you want to inspect results in a spreadsheet or pass them to a later analysis step. We will include both QC flags and basic sequence statistics.

In [23]:
report_path = output_dir / "sequence_qc_report.csv"

fieldnames = [
    "id", "description", "length", "gc_percent",
    "valid_symbols", "invalid_symbols", "passes_length", "passes"
]

with report_path.open("w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()

    for record in SeqIO.parse(fasta_path, "fasta"):
        seq = str(record.seq).upper()
        qc = sequence_qc(seq, min_length=10)
        canonical = sum(seq.count(base) for base in "ACGT")
        gc_count = seq.count("G") + seq.count("C")
        gc_percent = 100 * gc_count / canonical if canonical else None

        writer.writerow({
            "id": record.id,
            "description": record.description,
            "length": len(seq),
            "gc_percent": round(gc_percent, 3) if gc_percent is not None else "",
            "valid_symbols": qc["valid_symbols"],
            "invalid_symbols": ",".join(qc["invalid_symbols"]),
            "passes_length": qc["passes_length"],
            "passes": qc["passes"],
        })

print("CSV report saved to:", report_path)

CSV report saved to: outputs\sequence_qc_report.csv


## 10. Write and re-read a FASTA file

A useful validation step is to read the output file back in and check that the expected records are present. This is a simple form of round-trip testing.

In [24]:
round_trip_records = list(SeqIO.parse(filtered_fasta_path, "fasta"))

assert len(round_trip_records) == len(filtered_records)
assert [r.id for r in round_trip_records] == [r.id for r in filtered_records]

print("Round-trip check passed.")
print("IDs:", [record.id for record in round_trip_records])

Round-trip check passed.
IDs: ['SEQ001', 'SEQ002', 'SEQ005']


## 11. GenBank: richer annotations

GenBank is a structured format that can include sequence data, organism information, references, features, and feature locations. Unlike FASTA, it is designed to preserve much richer biological annotation.

The following example writes a minimal `SeqRecord` in GenBank format. GenBank output requires a molecule type annotation, so we explicitly provide it.

In [25]:
genbank_record = SeqRecord(
    Seq("ATGAAACCCGGGTAA"),
    id="SYNTHETIC_GB001",
    name="SYNTHETIC_GB001",
    description="Synthetic example for GenBank format practice",
)
genbank_record.annotations["molecule_type"] = "DNA"
genbank_record.annotations["topology"] = "linear"
genbank_record.annotations["data_file_division"] = "SYN"

genbank_path = output_dir / "synthetic_record.gb"
SeqIO.write([genbank_record], genbank_path, "genbank")

parsed_genbank = SeqIO.read(genbank_path, "genbank")
print("ID:", parsed_genbank.id)
print("Sequence:", parsed_genbank.seq)
print("Molecule type:", parsed_genbank.annotations.get("molecule_type"))
print("Topology:", parsed_genbank.annotations.get("topology"))

ID: SYNTHETIC_GB001
Sequence: ATGAAACCCGGGTAA
Molecule type: DNA
Topology: linear


### What about format conversion?

Biopython can convert between supported formats by reading records in one format and writing them in another. Conversion does not magically create missing metadata: a FASTA file converted to GenBank will not gain real organism annotations or gene features that were not present in the input.

For conversion, first check whether the input format contains the metadata you need.

In [26]:
# Example: convert the filtered FASTA records to another FASTA file.
# This demonstrates the read -> write pattern without inventing annotations.
converted_path = output_dir / "filtered_sequences_copy.fasta"

with filtered_fasta_path.open("r", encoding="utf-8") as source_handle:
    count = SeqIO.write(SeqIO.parse(source_handle, "fasta"), converted_path, "fasta")

print(f"Converted {count} records to {converted_path}")

Converted 3 records to outputs\filtered_sequences_copy.fasta


## 12. Streaming large files

For large datasets, avoid `list(SeqIO.parse(...))` unless you really need every record in memory. Iteration processes one record at a time.

```python
for record in SeqIO.parse("large_dataset.fasta", "fasta"):
    # Process this record, then continue to the next.
    print(record.id, len(record.seq))
```

If the workflow needs multiple passes, you can reopen the file or store only the small set of derived values you need.

## 13. Mini-project: build a reusable FASTA QC pipeline

The function below combines the main ideas from this notebook. It reads a FASTA file, calculates per-record metrics, writes records that pass the configured filter, and writes a CSV report.

It deliberately keeps the criteria transparent. Real pipelines should define and document thresholds based on their scientific use case.

In [27]:
def run_fasta_qc_pipeline(
    input_fasta,
    output_fasta,
    output_csv,
    min_length=10,
):
    """Filter FASTA records and write a per-record QC report."""
    input_fasta = Path(input_fasta)
    output_fasta = Path(output_fasta)
    output_csv = Path(output_csv)

    passing_records = []
    report_rows = []

    for record in SeqIO.parse(input_fasta, "fasta"):
        seq = str(record.seq).upper()
        qc = sequence_qc(seq, min_length=min_length)

        canonical_count = sum(seq.count(base) for base in "ACGT")
        gc_count = seq.count("G") + seq.count("C")
        gc_percent = 100 * gc_count / canonical_count if canonical_count else None

        report_rows.append({
            "id": record.id,
            "description": record.description,
            "length": len(seq),
            "gc_percent": round(gc_percent, 3) if gc_percent is not None else "",
            "valid_symbols": qc["valid_symbols"],
            "invalid_symbols": ",".join(qc["invalid_symbols"]),
            "passes_length": qc["passes_length"],
            "passes": qc["passes"],
        })

        if qc["passes"]:
            passing_records.append(record)

    output_fasta.parent.mkdir(parents=True, exist_ok=True)
    output_csv.parent.mkdir(parents=True, exist_ok=True)

    SeqIO.write(passing_records, output_fasta, "fasta")

    fieldnames = [
        "id", "description", "length", "gc_percent",
        "valid_symbols", "invalid_symbols", "passes_length", "passes"
    ]
    with output_csv.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(report_rows)

    return {
        "input_records": len(report_rows),
        "passing_records": len(passing_records),
        "output_fasta": str(output_fasta),
        "output_csv": str(output_csv),
    }


pipeline_result = run_fasta_qc_pipeline(
    input_fasta=fasta_path,
    output_fasta=output_dir / "pipeline_passed.fasta",
    output_csv=output_dir / "pipeline_report.csv",
    min_length=10,
)

pipeline_result

{'input_records': 5,
 'passing_records': 3,
 'output_fasta': 'outputs\\pipeline_passed.fasta',
 'output_csv': 'outputs\\pipeline_report.csv'}

## 14. Exercises

Try to solve these before looking at the suggested extensions.

### Exercise 1 — Record inspection
Read `synthetic_sequences.fasta` and print only each record's ID and length.

### Exercise 2 — Adjust the filter
Run the QC pipeline with `min_length=12`. How does the number of passing records change?

### Exercise 3 — Include ambiguous bases
Modify the QC logic so that `N` is allowed, but other non-canonical symbols are rejected. Report the fraction of `N` bases for each record.

### Exercise 4 — Add another metric
Add a `sequence_type` or `contains_start_codon` column to the CSV report. Explain why the presence of `ATG` alone does not prove that a sequence encodes a complete protein.

### Exercise 5 — Validate the output
After writing the filtered FASTA, read it again and assert that every record passes the same filter.

### Exercise 6 — Work with a real file
Use a FASTA file from a source you trust. Keep a copy of the original file, record its source and date, and document the filtering rules you apply.

## 15. Common pitfalls and good practices

- **Confusing `record.id` with the full header:** inspect `record.description` when the full FASTA header matters.
- **Assuming every file has one record:** use `SeqIO.parse()` for multi-record files.
- **Loading very large files into memory:** iterate over `SeqIO.parse()` when possible.
- **Treating ambiguous bases as automatically invalid:** the right policy depends on the downstream analysis.
- **Assuming format conversion restores missing information:** conversion can preserve available information, but cannot infer trustworthy annotations.
- **Overwriting source data:** write filtered or transformed records to a separate output path.
- **Using arbitrary QC thresholds as universal standards:** document why the thresholds are suitable for your task.
- **Forgetting provenance:** record dataset source, access date, format, and processing steps for reproducibility.

## Quick reference

| Task | Biopython pattern |
|---|---|
| Read many FASTA records | `SeqIO.parse(path, "fasta")` |
| Read exactly one record | `SeqIO.read(path, "fasta")` |
| Write records | `SeqIO.write(records, path, "fasta")` |
| Access sequence | `record.seq` |
| Access identifier | `record.id` |
| Access description | `record.description` |
| Access record length | `len(record.seq)` |
| Read GenBank | `SeqIO.parse(path, "genbank")` |
| Write GenBank | `SeqIO.write(records, path, "genbank")` |

## Summary

In this notebook, you learned how to move from individual Biopython sequence objects to real collections of sequence records. You read and wrote FASTA, inspected `SeqRecord` metadata, applied transparent filters, generated CSV summaries, and explored the richer annotations supported by GenBank.

The next logical lesson is **05 — Retrieving Biological Data from NCBI with `Bio.Entrez`**, where we will learn how to search for and retrieve records from public biological databases while respecting NCBI usage guidance and recording data provenance.